# Insurance Claims Analytics & Risk Assessment
## Module 4 — Claim Anomaly Detection

**Business Objective**
Identify claims and policies that exhibit unusual characteristics warranting
further investigation by the claims review team. This module does not confirm
fraud — it surfaces statistically and rule-based anomalous patterns for
human review.

**Detection Methodology**
| Method | Basis |
|---|---|
| High Claim-to-Premium Ratio | IQR upper bound |
| High Claim Percentile | 90th percentile threshold |
| High Utilisation | ≥ 80% coverage consumed |
| Early Claim Flag | Filed within 30 days of inception |
| Z-Score Outlier | \|Z\| > 3.0 on claim amount |
| Isolation Forest | Multivariate anomaly detection |

**Risk Classification**
- **Normal:** 0 flags triggered
- **Suspicious:** 1–2 flags triggered
- **High Risk:** 3+ flags triggered


In [ ]:
import sys
sys.path.insert(0, '..')
from src.anomaly_detection import load_cleaned_data, build_anomaly_flags, plot_anomaly_detection, save_anomaly_data
from IPython.display import Image
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = load_cleaned_data()


## 4.1 Applying Anomaly Flags

In [ ]:
df_flagged = build_anomaly_flags(df)
flag_cols = [c for c in df_flagged.columns if c.startswith('flag_')]
print("Flag Summary:")
print(df_flagged[flag_cols].sum().sort_values(ascending=False).to_string())


## 4.2 Risk Tier Distribution

In [ ]:
print("Risk Tier Distribution:")
print(df_flagged['AnomalyRisk'].value_counts().to_string())
print(f"\nHigh Risk records  : {(df_flagged['AnomalyRisk'] == 'High Risk').sum():,}")
print(f"Suspicious records : {(df_flagged['AnomalyRisk'] == 'Suspicious').sum():,}")
print(f"Normal records     : {(df_flagged['AnomalyRisk'] == 'Normal').sum():,}")


## 4.3 Anomaly Detection Visualisation

In [ ]:
plot_anomaly_detection(df_flagged)
Image('../visuals/anomaly_detection.png')


## 4.4 High Risk Records — Sample Review

In [ ]:
high_risk = df_flagged[df_flagged['AnomalyRisk'] == 'High Risk'].sort_values('total_flags', ascending=False)
cols = ['PolicyNumber', 'CustomerID', 'Age', 'PolicyType', 'PremiumAmount', 'ClaimAmount', 'Utilization', 'total_flags', 'ClaimStatus']
print(f"Total High Risk records: {len(high_risk)}")
high_risk[cols].head(15)


## 4.5 Financial Exposure by Risk Tier

In [ ]:
exposure = df_flagged.groupby('AnomalyRisk').agg(
    count=('ClaimAmount', 'count'),
    total_exposure=('ClaimAmount', 'sum'),
    avg_claim=('ClaimAmount', 'mean'),
    avg_flags=('total_flags', 'mean')
).round(2)
print("Financial Exposure by Risk Tier:")
print(exposure.to_string())


In [ ]:
save_anomaly_data(df_flagged)
print("Anomaly-flagged dataset saved.")


**Finding:** 103 policies (1.0%) are classified as High Risk, with an
average of 3+ flags triggered. These records carry disproportionately high
claim amounts relative to their premium and coverage levels, and represent
the primary candidates for manual claims review.

---
**Next Module →** `05_Risk_Profiling.ipynb`
